In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation

In [ ]:
import sys
!{sys.executable} -m pip install pandas numpy matplotlib --user

In [ ]:
df = pd.read_csv('BTC_DATA/btc_data.csv')
print(df.shape)
print(df.head())

In [ ]:
plt.rcParams['animation.embed_limit'] = 450  # allow up to 100MB

from IPython.display import HTML
from matplotlib.animation import FuncAnimation

# ── Data prep ─────────────────────────────────────────────────────────────────
df_anim = df[df['price_usd_close'] > 0].copy()
df_anim = df_anim.sort_values('block_height').reset_index(drop=True)

CYCLE = 210000
df_anim['theta'] = (df_anim['block_height'] % CYCLE) / CYCLE * 2 * np.pi
df_anim['r']     = np.log10(df_anim['price_usd_close'])

# ── Figure ────────────────────────────────────────────────────────────────────
plt.close('all')
fig = plt.figure(figsize=(16, 8), facecolor='black')
ax_s = fig.add_subplot(121, projection='polar')
ax_l = fig.add_subplot(122)

# ── Spiral ────────────────────────────────────────────────────────────────────
ax_s.set_facecolor('black')
ax_s.set_theta_zero_location('N')
ax_s.set_theta_direction(-1)

clock_labels = ['12','12:30','1','1:30','2','2:30','3','3:30',
                '4','4:30','5','5:30','6','6:30','7','7:30',
                '8','8:30','9','9:30','10','10:30','11','11:30']
ax_s.set_xticks(np.linspace(0, 2*np.pi, 24, endpoint=False))
ax_s.set_xticklabels(clock_labels, color='white', fontsize=8)

ax_s.set_yticks([0, 1, 2, 3, 4, 5, 6, 7])
ax_s.set_yticklabels(['$1','$10','$100','$1,000','$10,000',
                       '$100,000','$1,000,000','$10,000,000'],
                      color='white', fontsize=7)
ax_s.set_ylim(-0.5, 7.5)
ax_s.grid(color='white', alpha=0.3, linewidth=0.5)
ax_s.set_title('Spiral', color='white', fontsize=14, pad=15)

# ── Linear ────────────────────────────────────────────────────────────────────
ax_l.set_facecolor('black')
for spine in ax_l.spines.values():
    spine.set_color('white')
ax_l.tick_params(colors='white')
ax_l.set_xlabel('Block Height', color='white', fontsize=11)
ax_l.set_ylabel('Price (USD)', color='white', fontsize=11)
ax_l.set_title('Linear', color='white', fontsize=14, pad=15)
ax_l.set_xlim(df_anim['block_height'].min(), df_anim['block_height'].max())
ax_l.set_ylim(0, df_anim['price_usd_close'].max() * 1.1)
def price_formatter(x, p):
    if x >= 1000:   return f'${x:,.0f}'
    elif x >= 1:    return f'${x:.2f}'
    else:           return f'${x:.4f}'

ax_l.yaxis.set_major_formatter(plt.FuncFormatter(price_formatter))
# ── Lines + info text ─────────────────────────────────────────────────────────
spiral_line, = ax_s.plot([], [], color='orange', linewidth=0.8)
linear_line, = ax_l.plot([], [], color='orange', linewidth=0.8)
info_text = ax_l.text(0.02, 0.97, '', transform=ax_l.transAxes,
                       color='white', fontsize=11, verticalalignment='top',
                       fontfamily='monospace')

plt.tight_layout()
plt.subplots_adjust(wspace=0.35)

# ── Animation ─────────────────────────────────────────────────────────────────
STEP = 3  # rows per frame — lower = slower/smoother, higher = faster

def animate(frame):
    idx  = min(frame * STEP, len(df_anim) - 1)
    data = df_anim.iloc[:idx + 1]
    row  = df_anim.iloc[idx]
    spiral_line.set_data(data['theta'].values, data['r'].values)
    linear_line.set_data(data['block_height'].values, data['price_usd_close'].values)
    #y-scaling
    current_min = data['price_usd_close'].min()
    current_max = data['price_usd_close'].max()
    ax_l.set_ylim(current_min * 0.8, current_max * 1.2)
    ax_l.set_xlim(df_anim['block_height'].min(), data['block_height'].max() * 1.05)
    
    info_text.set_text(
        f"Price: ${row['price_usd_close']:,.2f}\n"
        f"Block Height: {int(row['block_height'])}\n"
        f"Date: {row['date']}"
    )
    return spiral_line, linear_line, info_text

frames = len(df_anim) // STEP + 1
anim   = FuncAnimation(fig, animate, frames=frames, interval=30, blit=False)

HTML(anim.to_jshtml())

In [ ]:
# ── Export as GIF ─────────────────────────────────────────────────────────────
from matplotlib.animation import PillowWriter

GIF_STEP = 30   # every 30th data point (fewer frames = smaller file)
GIF_FPS  = 20
GIF_DPI  = 80

plt.close('all')
fig_gif = plt.figure(figsize=(12, 6), facecolor='black')
ax_sg = fig_gif.add_subplot(121, projection='polar')
ax_lg = fig_gif.add_subplot(122)

ax_sg.set_facecolor('black')
ax_sg.set_theta_zero_location('N')
ax_sg.set_theta_direction(-1)
clock_labels = ['12','12:30','1','1:30','2','2:30','3','3:30',
                '4','4:30','5','5:30','6','6:30','7','7:30',
                '8','8:30','9','9:30','10','10:30','11','11:30']
ax_sg.set_xticks(np.linspace(0, 2*np.pi, 24, endpoint=False))
ax_sg.set_xticklabels(clock_labels, color='white', fontsize=7)
ax_sg.set_yticks([0, 1, 2, 3, 4, 5, 6, 7])
ax_sg.set_yticklabels(['$1','$10','$100','$1K','$10K','$100K','$1M','$10M'],
                       color='white', fontsize=6)
ax_sg.set_ylim(-0.5, 7.5)
ax_sg.grid(color='white', alpha=0.3, linewidth=0.5)
ax_sg.set_title('BTC Price Spiral', color='white', fontsize=12, pad=15)

ax_lg.set_facecolor('black')
for spine in ax_lg.spines.values():
    spine.set_color('white')
ax_lg.tick_params(colors='white')
ax_lg.set_xlabel('Block Height', color='white', fontsize=10)
ax_lg.set_ylabel('Price (USD)', color='white', fontsize=10)
ax_lg.set_title('BTC Price (Linear)', color='white', fontsize=12, pad=15)

def price_formatter_gif(x, p):
    if x >= 1000:  return f'${x:,.0f}'
    elif x >= 1:   return f'${x:.2f}'
    else:          return f'${x:.4f}'
ax_lg.yaxis.set_major_formatter(plt.FuncFormatter(price_formatter_gif))

spiral_line_g, = ax_sg.plot([], [], color='orange', linewidth=0.8)
linear_line_g, = ax_lg.plot([], [], color='orange', linewidth=0.8)
info_g = ax_lg.text(0.02, 0.97, '', transform=ax_lg.transAxes,
                    color='white', fontsize=10, verticalalignment='top',
                    fontfamily='monospace')

plt.tight_layout()
plt.subplots_adjust(wspace=0.35)

def animate_gif(frame):
    idx  = min(frame * GIF_STEP, len(df_anim) - 1)
    data = df_anim.iloc[:idx + 1]
    row  = df_anim.iloc[idx]
    spiral_line_g.set_data(data['theta'].values, data['r'].values)
    linear_line_g.set_data(data['block_height'].values, data['price_usd_close'].values)
    ax_lg.set_ylim(data['price_usd_close'].min() * 0.8,
                   data['price_usd_close'].max() * 1.2)
    ax_lg.set_xlim(df_anim['block_height'].min(),
                   data['block_height'].max() * 1.05)
    info_g.set_text(
        f"Price: ${row['price_usd_close']:,.2f}\n"
        f"Block: {int(row['block_height'])}\n"
        f"Date:  {row['date']}"
    )
    return spiral_line_g, linear_line_g, info_g

frames_gif = len(df_anim) // GIF_STEP + 1
anim_gif = FuncAnimation(fig_gif, animate_gif, frames=frames_gif,
                         interval=1000//GIF_FPS, blit=False)

gif_path = '/Users/johnnyreed/Desktop/Bitcoin Spiral/bitcoin_spiral.gif'
print(f"Rendering {frames_gif} frames... this will take a few minutes ⏳")
anim_gif.save(gif_path, writer=PillowWriter(fps=GIF_FPS), dpi=GIF_DPI)
print(f"✓ Done! Saved to {gif_path}")
plt.close(fig_gif)